# mac_nxn_array bring-up on PYNQ-Z2

Runs the 8x8 LNS matrix-multiply kernel (`mac_nxn_array`, from
`silicon/src/mac_silicon.cpp`) against the test vectors generated on the host
by `silicon/host/gen_vectors.cpp`. Same matrices, same golden model, same
seeds as the cosim testbench (`silicon/tb/mac_nxn_cosim_tb.cpp`) -- the point
of this notebook is a bit-exact hardware-vs-software check, not a fresh
statistical run.

Expects, copied alongside this notebook (see `silicon/board/README.md`):
- `mac_lns.bit` + `mac_lns.hwh` (same basename, same directory)
- `vectors/` (the 30 `case*.bin` files + `manifest.json` from `gen_vectors`)

Each LNS element is a fixed 5-byte struct
`[sign][zero][exponent][quotient][remainder]`, one byte per field. An 8x8
matrix is `8*8*5 = 320` bytes, row-major -- that is the whole DMA payload per
operand.


## 1. Load the overlay

Load the bitstream and grab the IP handle. The instance name PYNQ assigns
usually matches the HLS top function name with a `_0` suffix
(`mac_nxn_array_0`), but block-design renames happen -- if the direct
attribute isn't there, fall back to listing `ol.ip_dict` so you can see the
actual name and fix the one line below.


In [ ]:
from pynq import Overlay, allocate
import numpy as np
import json, time, os

ol = Overlay('mac_lns.bit')

try:
    ip = ol.mac_nxn_array_0
except AttributeError:
    print("Default IP name 'mac_nxn_array_0' not found. Available IPs:")
    for name in ol.ip_dict.keys():
        print(" -", name)
    raise


## 2. Allocate DMA buffers

Three physically-contiguous 320-byte buffers (one 8x8 LNS matrix each):
operand A, operand B, and the result. `pynq.allocate` returns a numpy array
backed by CMA memory, so we can fill it with `np.fromfile` and hand its
`.physical_address` straight to the AXI-Lite pointer registers.


In [ ]:
MATRIX_BYTES = 320  # 8*8*5

buf_a = allocate(shape=(MATRIX_BYTES,), dtype=np.uint8)
buf_b = allocate(shape=(MATRIX_BYTES,), dtype=np.uint8)
buf_r = allocate(shape=(MATRIX_BYTES,), dtype=np.uint8)

print("buffers allocated:")
print(" a:", hex(buf_a.physical_address))
print(" b:", hex(buf_b.physical_address))
print(" r:", hex(buf_r.physical_address))


## 3. `run_case` helper

Standard HLS `s_axilite` block-level control protocol:

| offset | field |
|---|---|
| 0x00 | `ap_ctrl` -- bit0 `ap_start`, bit1 `ap_done`, bit2 `ap_idle`, bit7 `auto_restart` |
| 0x10/0x14 | `array_input_a` physical address, lo/hi 32 bits |
| 0x1c/0x20 | `array_input_b` physical address, lo/hi 32 bits |
| 0x28/0x2c | `result` physical address, lo/hi 32 bits |

Load the two operand `.bin` files into the buffers, flush them to DDR
(`sync_to_device`), point the three `m_axi` pointers at the buffers, pulse
`ap_start`, poll for done/idle, then pull the result back
(`sync_from_device`) and compare byte-for-byte against `case<i>_expected.bin`.
Exact match is the bar here -- the LNS grid plus a bit-exact HLS datapath
means there is no tolerance band on hardware, unlike the float-vs-quantized
tolerance the *cosim* testbench applies against the unquantized golden
matmul.


In [ ]:
AP_CTRL      = 0x00
A_ADDR_LO    = 0x10
A_ADDR_HI    = 0x14
B_ADDR_LO    = 0x1c
B_ADDR_HI    = 0x20
R_ADDR_LO    = 0x28
R_ADDR_HI    = 0x2c

AP_START = 1 << 0
AP_DONE  = 1 << 1
AP_IDLE  = 1 << 2

VECTORS_DIR = 'vectors'

def _write_ptr(lo_off, hi_off, phys_addr):
    ip.write(lo_off, phys_addr & 0xFFFFFFFF)
    ip.write(hi_off, (phys_addr >> 32) & 0xFFFFFFFF)

def launch_and_wait(timeout_s=1.0):
    """Pulse ap_start and poll until ap_done or ap_idle. Returns elapsed seconds."""
    t0 = time.perf_counter()
    ip.write(AP_CTRL, AP_START)
    deadline = t0 + timeout_s
    while True:
        status = ip.read(AP_CTRL)
        if (status & AP_DONE) or (status & AP_IDLE):
            break
        if time.perf_counter() > deadline:
            raise TimeoutError(f"kernel did not finish within {timeout_s}s (ap_ctrl=0x{status:02x})")
    return time.perf_counter() - t0

def run_case(name, vectors_dir=VECTORS_DIR, verbose=True):
    """Load case<name> operands, run the kernel once, check the result.

    Returns True on an exact byte-for-byte match, False otherwise.
    """
    a_path = os.path.join(vectors_dir, f'{name}_a.bin')
    b_path = os.path.join(vectors_dir, f'{name}_b.bin')
    exp_path = os.path.join(vectors_dir, f'{name}_expected.bin')

    a_bytes = np.fromfile(a_path, dtype=np.uint8)
    b_bytes = np.fromfile(b_path, dtype=np.uint8)
    expected = np.fromfile(exp_path, dtype=np.uint8)
    assert a_bytes.size == MATRIX_BYTES and b_bytes.size == MATRIX_BYTES and expected.size == MATRIX_BYTES, \
        f"{name}: vector file is not {MATRIX_BYTES} bytes"

    buf_a[:] = a_bytes
    buf_b[:] = b_bytes
    buf_a.sync_to_device()
    buf_b.sync_to_device()

    _write_ptr(A_ADDR_LO, A_ADDR_HI, buf_a.physical_address)
    _write_ptr(B_ADDR_LO, B_ADDR_HI, buf_b.physical_address)
    _write_ptr(R_ADDR_LO, R_ADDR_HI, buf_r.physical_address)

    elapsed = launch_and_wait()

    buf_r.sync_from_device()
    ok = np.array_equal(buf_r, expected)

    if verbose:
        status = 'PASS' if ok else 'FAIL'
        print(f"{name}: {status}  ({elapsed*1e6:.1f} us wall, incl. Python/poll overhead)")
        if not ok:
            mism = np.nonzero(buf_r != expected)[0]
            print(f"  first mismatched byte offset(s): {mism[:10].tolist()}")

    return ok


## 4. Run all cases

Reads `manifest.json` (written by `gen_vectors`) so the case list and count
live in one place instead of being hardcoded here.


In [ ]:
with open(os.path.join(VECTORS_DIR, 'manifest.json')) as f:
    manifest = json.load(f)

results = {}
for case in manifest['cases']:
    name = case['name']
    results[name] = run_case(name)

n_pass = sum(results.values())
n_total = len(results)
print(f"\n{n_pass}/{n_total} cases PASSED")
if n_pass != n_total:
    failed = [k for k, v in results.items() if not v]
    print("FAILED:", failed)


## 5. Timing

Launch case 0 in a tight loop (rewriting `ap_start` and polling `ap_done`
each time, buffers already loaded from the run above) and report the mean
wall-clock time per launch.

The RTL cosim/synthesis estimate is ~2979 cycles @ 100 MHz ~= 30 us per
matmul (m_axi transfers included). Expect the measured number here to be
noticeably higher: this loop pays Python-level MMIO register read/write
overhead on every poll iteration, which dominates at this problem size --
that gap is expected and is not a hardware regression. It is not a clean
per-invocation hardware measurement (no AXI trace, no ILA), just a
software-observable sanity number.


In [ ]:
N_LAUNCHES = 1000

# Buffers already hold case0's operands from the correctness run above;
# re-point the pointer registers once, then just re-pulse ap_start.
_write_ptr(A_ADDR_LO, A_ADDR_HI, buf_a.physical_address)
_write_ptr(B_ADDR_LO, B_ADDR_HI, buf_b.physical_address)
_write_ptr(R_ADDR_LO, R_ADDR_HI, buf_r.physical_address)

t0 = time.perf_counter()
for _ in range(N_LAUNCHES):
    ip.write(AP_CTRL, AP_START)
    while not (ip.read(AP_CTRL) & AP_DONE):
        pass
t1 = time.perf_counter()

mean_us = (t1 - t0) / N_LAUNCHES * 1e6
print(f"{N_LAUNCHES} launches, mean {mean_us:.1f} us/launch (Python register-poll overhead included)")
print("RTL cosim/synthesis estimate: ~30 us/launch (2979 cycles @ 100 MHz)")
print(f"ratio (measured / RTL estimate): {mean_us/30.0:.1f}x")


### Reading the timing number

If the measured mean is well above ~30 us, that is expected and not a sign
of a broken kernel: each iteration here does a Python function call, an MMIO
write, then a tight Python `while` loop doing MMIO reads over the AXI-Lite
control bus -- register access from Python through the PYNQ MMIO layer costs
microseconds per transaction on its own, and this loop does several per
launch. To see hardware-only latency you would need an AXI performance
monitor / ILA capture, which is out of scope for this notebook. The
correctness check in section 4 (bit-exact match against `mac_nxn_array`'s
software golden output) is the meaningful result here; this timing cell is a
sanity/order-of-magnitude check, not a latency benchmark.
